# 01 · Train the player, referee, ball and hoop detector on several datasets

The tracking dataset needs every on-court player and the ball in every frame of an NBA broadcast.
This notebook trains the YOLO detector for that on **several datasets at once** and scores it
**on each dataset separately**, so you can see where it holds up and where it doesn't.

| Source | What it adds | Role |
|---|---|---|
| `v17` | your Roboflow set (workspace-5ujvu, v17): all four classes, mostly full-court frames | train |
| `gamedet` | Basketball Game Detection (deeplearning-gmyc6): NBA broadcast frames, all four classes | train |
| `broadcast` | Basketball Broadcast (basketball-w376e): broadcast frames, all four classes | train |
| `sportsmot` | SportsMOT basketball clips with every player boxed: crowds, overlaps, blur | train |
| `own` | your broadcast frames from notebook 03, corrected in Roboflow | train (weighted x3) |
| `own_test` | frames of clips you keep back to score on | test only |
| `negatives` | crowd, bench, replays, ads: nothing to box | train only |

Anything missing is skipped. Classes: `Player`, `Ref`, `Ball`, `Hoop`; other classes in a dataset
(clock, scoreboard, backboard) are dropped.

**How leakage is kept out** (`courtvision/training/sources.py`):

* Each dataset's own train/valid/test split is ignored: Roboflow splits frames at random, so
  neighbouring frames of one clip and augmented copies of one frame land on both sides.
* Images are tied together when they come from the same video (file names), the same SportsMOT
  game, or look the same (a 64-bit image hash), **across datasets too**: a frame re-exported by
  another dataset is tied to its original. Tied images always share a split.
* `own_test` pulls everything tied to it into test, so nothing near it is trained on.
* Each source is scored only on the classes it labels itself. SportsMOT's referee, ball and hoop
  boxes come from the current detector, so scoring against them would measure agreement with the
  old model.
* The new-vs-current comparison uses only test images the current detector cannot have seen.

**Cross-validation:** leave-one-dataset-out (train without a dataset, test on it) shows whether
the detector generalises to footage it has never seen, which is what a new NBA game is. Grouped
k-fold is optional.

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"   # everything that must survive a disconnect lives here
for d in ["models", "runs", "reports", "training_data", "input_videos", "output"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # to test a pull request, put its branch name here, then Runtime > Restart session > Run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models,train]"
import sys
sys.path.insert(0, "/content/courtvision")   # importable now, without restarting the session

## 1 · Download the datasets

Store your Roboflow key in Colab **Secrets** as `ROBOFLOW_API_KEY` (and the Hugging Face token as
`HF_TOKEN` for SportsMOT). `v17` is pinned so results stay comparable; the two new sets take
their latest version, which is printed.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))   # Colab Secrets (key icon). Never paste a key into a cell.

def download(workspace, project, version=None):
    """A Roboflow dataset in YOLO format; version None means the latest."""
    p = rf.workspace(workspace).project(project)
    version = version or max(int(str(v.version).split("/")[-1]) for v in p.versions())
    d = p.version(version).download("yolov8", location=f"/content/datasets/{project}-v{version}")
    print(f"{workspace}/{project} v{version} -> {d.location}")
    return d.location

ROOTS = {
    "v17": download("workspace-5ujvu", "basketball-players-fy4c2-vfsuv", 17),
    "gamedet": download("deeplearning-gmyc6", "basketball-game-detection"),
    "broadcast": download("basketball-w376e", "basketball-broadcast"),
}

### SportsMOT basketball (optional, once access is granted on Hugging Face)

The full download is about 23 GB (three sports, and a test split without labels), too big for
Drive. The first run keeps only the basketball train and val clips, in one zip on Drive
(`training_data/sportsmot_basketball.zip`), taken from a download already in
`training_data/sportsmot` if there is one, else downloaded to Colab's disk. Every later run just
unpacks that zip to Colab's disk, which is also much faster to read than Drive.

SportsMOT boxes players only, so referees, the ball and the hoop are filled in from the current
detector's confident boxes, and a frame where it is unsure about one of them (a box between 0.15
and 0.5) is skipped rather than half-labelled. One frame in 10 is used (neighbouring frames are
near-identical). Frames are named `game__clip_frame` so a whole game stays on one side of every
split: SportsMOT's own val clips can come from its train games.

In [ ]:
import shutil
from pathlib import Path
from courtvision.training import sportsmot

SM_ZIP = f"{DRIVE_ROOT}/training_data/sportsmot_basketball.zip"     # basketball only, made once
SM_FOLDER = f"{DRIVE_ROOT}/training_data/sportsmot"                # or a SportsMOT download already in Drive
LOCAL_ZIP = "/content/sportsmot_basketball.zip"
if not os.path.exists(SM_ZIP) and not os.path.exists(LOCAL_ZIP):
    try:
        raw = SM_FOLDER if os.path.isdir(SM_FOLDER) else "/content/sportsmot_raw"
        if raw == "/content/sportsmot_raw":   # nothing in Drive yet: download ~23 GB to Colab's disk
            from huggingface_hub import snapshot_download
            snapshot_download("MCG-NJU/SportsMOT", repo_type="dataset", token=userdata.get("HF_TOKEN"), local_dir=raw)
        print(sportsmot.basketball_zip(raw, LOCAL_ZIP))
        if os.path.getsize(LOCAL_ZIP) < shutil.disk_usage(DRIVE_ROOT).free:
            shutil.copy(LOCAL_ZIP, SM_ZIP)     # next session starts from the zip
        if raw == "/content/sportsmot_raw":
            shutil.rmtree(raw)
    except Exception as e:
        print("SportsMOT not available:", e)
zip_path = SM_ZIP if os.path.exists(SM_ZIP) else LOCAL_ZIP
SM_DATA = sportsmot.unpack(zip_path, "/content/sportsmot") if os.path.exists(zip_path) else None
if SM_DATA:
    print({split: len(sportsmot.basketball_sequences(SM_DATA, split)) for split in ["train", "val"]}, "basketball sequences")

In [ ]:
import numpy as np
from ultralytics import YOLO
from courtvision.training import sportsmot
from courtvision.training.labels import fill_missing
from courtvision.training.sources import DETECT_CLASSES, class_map

CURRENT = f"{DRIVE_ROOT}/models/player_detector.pt"
SM_YOLO = "/content/datasets/sportsmot-yolo"
if SM_DATA and not os.path.exists(f"{SM_YOLO}/data.yaml"):
    teacher = YOLO(CURRENT)
    to_ours = class_map([teacher.names[i] for i in sorted(teacher.names)], DETECT_CLASSES)  # teacher id -> our id
    fill = [DETECT_CLASSES.index(c) for c in ["Ref", "Ball", "Hoop"]]

    def fill_from_teacher(img, players):
        b = teacher.predict(img, imgsz=1280, conf=0.15, verbose=False)[0].boxes
        cls = np.array([to_ours.get(int(c), -1) for c in b.cls.cpu().numpy()], int)
        return fill_missing(players, np.zeros(len(players), int), b.xyxy.cpu().numpy(), cls,
                            b.conf.cpu().numpy(), fill_classes=fill)

    print(sportsmot.to_yolo(SM_DATA, SM_YOLO, DETECT_CLASSES, stride=10, fill=fill_from_teacher))

## 2 · Build one leakage-free dataset

`weight` repeats a source's training images (never validation or test): your own frames count
three times so a few hundred of them are not drowned out. The leakage checks must all be 0; the
table shows images and boxes per class for each source and split (a class with under ~100 test
boxes in a source gives a noisy score for it). `largest_component_share` is the biggest group of
tied images as a share of its source: above ~0.3, that source's splits are coarse.

In [ ]:
import pandas as pd
from courtvision.training.sources import Source, build, leakage_report, summary

TD = f"{DRIVE_ROOT}/training_data"
SOURCES = [Source("v17", ROOTS["v17"]), Source("gamedet", ROOTS["gamedet"]), Source("broadcast", ROOTS["broadcast"])]
if os.path.exists(f"{SM_YOLO}/data.yaml"):
    SOURCES.append(Source("sportsmot", SM_YOLO, group_by="prefix"))
if os.path.exists(f"{TD}/own_labelled/data.yaml"):
    SOURCES.append(Source("own", f"{TD}/own_labelled", weight=3))
if os.path.exists(f"{TD}/own_test/data.yaml"):
    SOURCES.append(Source("own_test", f"{TD}/own_test", role="test"))
if os.path.isdir(f"{TD}/negatives"):
    SOURCES.append(Source("negatives", f"{TD}/negatives", role="negatives"))

DATA = "/content/datasets/detector"
manifest = build(SOURCES, DETECT_CLASSES, DATA, val_frac=0.15, test_frac=0.15, seed=0)
report = leakage_report(manifest)
print(report)
assert report["components_in_two_splits"] == report["groups_in_two_splits"] == report["test_source_images_not_in_test"] == 0
manifest.to_csv(f"{DRIVE_ROOT}/reports/detector_manifest.csv", index=False)

# Classes each source labels itself. SportsMOT's referees, balls and hoops came from the current detector.
PSEUDO = {"sportsmot": ["Ref", "Ball", "Hoop"]}
SCORED = {s: [c for c in DETECT_CLASSES if g[c].sum() > 0 and c not in PSEUDO.get(s, [])]
          for s, g in manifest.groupby("source") if s != "negatives"}
print("scored classes:", SCORED)
summary(manifest, DETECT_CLASSES)

Look at each source before training on it. Every player on the court should be boxed: a dataset
that skips players teaches the model that players are background. Grids are saved to
`reports/look_<source>.jpg`.

In [ ]:
from courtvision.training.look import show_grid
for src in manifest.source.unique():
    show_grid(manifest[manifest.source == src].image.tolist(), DETECT_CLASSES, title=src,
              save=f"{DRIVE_ROOT}/reports/look_{src}.jpg")

## 3 · Scoring and training helpers

Scores at the thresholds the pipeline runs at (`configs/default.yaml`), per frame, because the
tracking dataset needs every player in every frame:

* `player_recall_per_frame`: players found / players labelled, averaged over frames
* `all_players_found_share`: frames where every labelled player was found
* `Ball_recall`, `Ball_precision`, `phantom_ball_frame_share` (frames with a ball box where there is none)

Runs are saved in Drive and resume after a disconnect. Change `RUN_TAG` whenever the sources
change, so an old finished run isn't reused.

In [ ]:
import yaml
from courtvision.config import Config
from courtvision.training.evaluate import score_images, summarise

RUN_TAG = "v1"
CFG = Config.load("/content/courtvision/configs/default.yaml")
CONF = {"Player": CFG.detection.conf, "Ref": CFG.detection.conf, "Hoop": CFG.detection.conf, "Ball": CFG.detection.ball_conf}
KEY = ["frames", "player_recall_per_frame", "all_players_found_share", "Player_precision",
       "Ball_recall", "Ball_precision", "phantom_ball_frame_share", "Ref_recall", "Hoop_recall"]

def images(split, sources=None, where=None):
    """{source: image paths} of one split; where: optional extra row filter on the manifest."""
    m = manifest[(manifest.split == split) & manifest.source.isin(list(sources or SCORED))]
    if where is not None:
        m = m[where.loc[m.index]]
    return {s: g.image.tolist() for s, g in m.groupby("source")}

def score(weights, by_source, imgsz=1280):
    """One row per source, scored on the classes that source labels itself."""
    model = YOLO(weights)
    rows = {s: summarise(score_images(model, imgs, DETECT_CLASSES, {c: CONF[c] for c in SCORED[s]}, imgsz=imgsz))
            for s, imgs in by_source.items() if imgs}
    return pd.DataFrame(rows).T.reindex(columns=KEY)

def train(name, data_yaml, base, **args):
    """Train, or resume after a disconnect; returns the best weights. Finished runs are reused."""
    run = f"{DRIVE_ROOT}/runs/{name}"
    if not os.path.exists(f"{run}/done"):
        last = f"{run}/weights/last.pt"
        if os.path.exists(last):
            try:
                YOLO(last).train(resume=True)
            except AssertionError as e:   # it had finished; only the marker was missing
                print(e)
        else:
            YOLO(base).train(data=data_yaml, project=f"{DRIVE_ROOT}/runs", name=name, exist_ok=True, **args)
        open(f"{run}/done", "w").close()
    return f"{run}/weights/best.pt"

## 4 · Cross-dataset validation: leave one dataset out

For each training source: a small model trained **with** it (on everything) and one trained
**without** it (and without anything tied to it), both scored on that source's test split. Short
runs on a small model: the point is the comparison, not the best weights.

* A small gap means the other datasets already cover that footage.
* A big gap means that dataset brings something the others lack. If `gamedet` or `broadcast`
  drops a lot when held out, broadcast frames are what the model needs more of, and your own
  labelled frames (notebook 03) are the next step.

About 15 to 25 minutes per run on an A100. Set `RUN_LODO = False` to skip.

In [ ]:
from courtvision.training.sources import lodo_folds

RUN_LODO = True
CV_BASE = "yolo11s.pt"
CV = dict(imgsz=960, epochs=30, patience=10, batch=-1, cos_lr=True, close_mosaic=5, plots=False)
folds = lodo_folds(DATA, min_images=50) if RUN_LODO else {}   # sources with 50+ images
if folds:
    with_it = score(train(f"det_{RUN_TAG}_cv_all", f"{DATA}/data.yaml", CV_BASE, **CV),
                    images("test", list(folds)), CV["imgsz"])
    without = pd.concat([score(train(f"det_{RUN_TAG}_cv_without_{held}", str(y), CV_BASE, **CV),
                               images("test", [held]), CV["imgsz"]) for held, y in folds.items()])
    cols = ["player_recall_per_frame", "all_players_found_share", "Ball_recall", "phantom_ball_frame_share"]
    lodo = pd.concat({"with": with_it[cols], "without": without[cols]}, axis=1).swaplevel(axis=1)[cols]
    lodo.to_csv(f"{DRIVE_ROOT}/reports/detector_lodo_{RUN_TAG}.csv")
    display(lodo)

### Optional: grouped k-fold

Classic cross-validation over the pooled training data, whole videos per fold, scored per source.
The spread across folds says how far a single validation score can be trusted. Off by default
(K more runs).

In [ ]:
from courtvision.training.sources import kfold_folds

RUN_KFOLD, K = False, 3
if RUN_KFOLD:
    rows = []
    for i, fold_yaml in enumerate(kfold_folds(DATA, k=K)):
        best = train(f"det_{RUN_TAG}_kfold{i}", str(fold_yaml), CV_BASE, **CV)
        val = set(open(yaml.safe_load(open(fold_yaml))["val"]).read().split())
        by_source = {s: [p for p in g.image if p in val] for s, g in manifest[manifest.source.isin(SCORED)].groupby("source")}
        rows.append(score(best, by_source, CV["imgsz"]).assign(fold=i))
    kfold = pd.concat(rows)
    display(kfold.groupby(level=0)[["player_recall_per_frame", "Ball_recall", "phantom_ball_frame_share"]].agg(["mean", "std"]))

## 5 · Train on everything

Broadcast balls are 15 to 30 px, so the final model trains at `imgsz=1280`. Early stopping uses
the pooled validation split; test is never touched.

In [ ]:
FINAL = dict(imgsz=1280, epochs=150, patience=30, batch=-1, cos_lr=True, close_mosaic=15, save_period=10)
NEW = train(f"det_{RUN_TAG}_final_yolo11m_1280", f"{DATA}/data.yaml", "yolo11m.pt", **FINAL)

## 6 · New model vs current model, per source

Each source's test split. The current detector was trained on v17 with v17's own random split, so
it may have seen any v17 test image, or a copy of one in another dataset. The comparison that
decides (`fair`) uses only test images not tied to any v17 image; the `v17` rows are shown for
reference only.

In [ ]:
v17_components = set(manifest[manifest.source == "v17"].component)
unseen_by_current = ~manifest.component.isin(v17_components)
DECIDE_ON = [s for s in ["own_test", "gamedet", "broadcast", "sportsmot", "own"] if s in SCORED]

fair = images("test", DECIDE_ON, where=unseen_by_current)
table = pd.concat({"new": score(NEW, fair), "current": score(CURRENT, fair)}).swaplevel().sort_index()
reference = pd.concat({"new": score(NEW, images("test", ["v17"])), "current": score(CURRENT, images("test", ["v17"]))})
table.to_csv(f"{DRIVE_ROOT}/reports/detector_new_vs_current_{RUN_TAG}.csv")
display(table)
display(reference)

## 7 · Tracking on held-out SportsMOT games

A court position is only useful if it stays attached to one player, so both detectors also run
through the pipeline's tracker (camera-compensated ByteTrack, `configs/default.yaml` settings) on
the full 25 fps sequences of the SportsMOT games the build put in test. **MOTA** counts misses,
false boxes and identity switches against all labelled boxes; **IDF1** is how much of each
player's time stays under one id. Referees are left out on both sides (SportsMOT boxes only
players).

In [ ]:
import cv2
from courtvision import camera, track
from courtvision.detect import detect_objects
from courtvision.training.evaluate import mot_metrics, mot_table

MAX_SEQS = 8   # a few hundred frames each

def track_sequence(weights, seq):
    """{frame: {track id: box}} for players, as the pipeline tracks them."""
    paths = sorted((seq / "img1").glob("*.jpg"))
    frames = lambda: (cv2.imread(str(p)) for p in paths)
    dets = detect_objects(frames(), weights, CFG.detection, total=len(paths))
    people = dets[dets.role.isin(["player", "referee"])].groupby("frame")
    boxes = [people.get_group(i)[["x1", "y1", "x2", "y2"]].values.tolist() if i in people.groups else [] for i in range(len(paths))]
    motion, cuts = camera.estimate_motion(frames(), boxes, total=len(paths))
    tracks = track.track_people(dets, len(paths), sportsmot.seq_fps(seq), motion, cuts, CFG.tracking.track_buffer)
    players = tracks[tracks.role == "player"]
    return {int(paths[f].stem): {int(r.track_id): [r.x1, r.y1, r.x2, r.y2] for r in g.itertuples()}
            for f, g in players.groupby("frame")}

mot = {}
if "sportsmot" in SCORED:
    test_games = {Path(p).stem.split("__")[0] for p in images("test", ["sportsmot"]).get("sportsmot", [])}
    seqs = sportsmot.sequences_of_games(SM_DATA, test_games)[:MAX_SEQS]
    print(len(test_games), "test games;", "sequences:", [s.name for s in seqs])
    for label, w in [("new", NEW), ("current", CURRENT)]:
        mot[label] = mot_table({s.name: mot_metrics(sportsmot.read_tracks(s), track_sequence(w, s)) for s in seqs})
    tracking = pd.concat({k: v.loc[["overall"]] for k, v in mot.items()})
    tracking.to_csv(f"{DRIVE_ROOT}/reports/detector_tracking_{RUN_TAG}.csv")
    display(tracking)

## 8 · Keep the new detector only if it is better

On every source in the fair comparison the new model must find at least as many players per
frame and balls (within 0.01), without more phantom-ball frames, and track at least as well
(IDF1). The previous file is kept as `player_detector_prev.pt`.

In [ ]:
from courtvision.training.evaluate import compare

verdict = {}
for src in table.index.get_level_values(0).unique():
    ok, why = compare(table.loc[(src, "new")], table.loc[(src, "current")],
                      higher=["player_recall_per_frame", "Ball_recall"], lower=["phantom_ball_frame_share"])
    verdict[src] = "ok" if ok else "; ".join(why)
if mot:
    ok, why = compare(mot["new"].loc["overall"], mot["current"].loc["overall"], higher=["idf1", "mota"])
    verdict["tracking"] = "ok" if ok else "; ".join(why)
for k, v in verdict.items():
    print(f"{k:12s} {v}")

if os.path.exists(CURRENT) and any(v != "ok" for v in verdict.values()):
    print("\nKept the current detector. The new one is at", NEW)
else:
    if os.path.exists(CURRENT):
        shutil.copy(CURRENT, f"{DRIVE_ROOT}/models/player_detector_prev.pt")
    shutil.copy(NEW, CURRENT)
    print("\nSaved", CURRENT)